# Appendix. 04 - Data construction and linear regression

This notebook combines the former **Appendix C(4)** and **Appendix C(5)** workflows into one reproducible pipeline.

The workflow:

1. Computes overall **Origin-SRMSE** and **Destination-SRMSE** from the common HTS and LCS OD matrices.
2. Merges the spatial and survey attributes used in the regression analysis.
3. Constructs the origin- and destination-level regression tables.
4. Applies PCA to the four standardized land-use variables.
5. Estimates the baseline and interaction OLS models reported in the manuscript.
6. Checks the principal results against Table 5 and Appendix B.2.

LCS is treated as the reference dataset. For each origin or destination, SRMSE is computed across all 424 counterpart zones in the common OD frame, including zero-flow cells.

In [ ]:
from pathlib import Path

from IPython.display import display

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# =========================================================
# Project paths
# =========================================================
BASE_DIR = Path.cwd()

OD_DATA_DIR = BASE_DIR / "2_data_OD"
REGRESSION_DATA_DIR = BASE_DIR / "4_data_regression"
REGRESSION_RESULT_DIR = BASE_DIR / "5_Result_Regression"

REGRESSION_DATA_DIR.mkdir(parents=True, exist_ok=True)
REGRESSION_RESULT_DIR.mkdir(parents=True, exist_ok=True)

# OD inputs created in Appendix C(1) and C(2)
OD_HTS_FILE = OD_DATA_DIR / "OD_HTS.csv"
OD_LCS_FILE = OD_DATA_DIR / "OD_LCS.csv"

# Spatial and survey attributes
ORIGIN_ATTR_FILE = REGRESSION_DATA_DIR / "spatial_attribute_(seoul).csv"
DESTINATION_ATTR_FILE = (
    REGRESSION_DATA_DIR / "spatial_attribute_(seoul)_Destination.csv"
)

# Intermediate SRMSE outputs
ORIGIN_SRMSE_FILE = REGRESSION_DATA_DIR / "origin_srmse_overall.csv"
DESTINATION_SRMSE_FILE = (
    REGRESSION_DATA_DIR / "destination_srmse_overall.csv"
)

# Regression-ready outputs
ORIGIN_REGRESSION_FILE = REGRESSION_RESULT_DIR / "df_origin.csv"
DESTINATION_REGRESSION_FILE = REGRESSION_RESULT_DIR / "df_destination.csv"

N_SEOUL_ZONES = 424
INCOME_LEVELS = [
    "5_class",
    "6_class",
    "7_class",
    "8_class",
    "9_class",
    "10_class",
]
INCOME_REFERENCE = "10_class"


## 1. Helper functions for OD-based SRMSE

In [ ]:
def read_csv_flexible(path: Path) -> pd.DataFrame:
    """Read a project CSV file using common encodings."""
    for encoding in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Unable to read file with supported encodings: {path}")


def standardize_zone_id(series: pd.Series) -> pd.Series:
    """Convert zone IDs to stable string keys."""
    numeric = pd.to_numeric(series, errors="coerce").astype("Int64")
    if numeric.isna().any():
        raise ValueError(
            f"Zone ID contains {numeric.isna().sum():,} missing or invalid values."
        )
    return numeric.astype("string")


def standardize_od_table(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize OD table columns for stable merging."""
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()

    rename_map = {
        "출발행정동코드": "origin_id",
        "도착행정동코드": "destination_id",
        "통행횟수": "trips",
        "통행량": "trips",
    }
    out = out.rename(columns=rename_map)

    required = ["origin_id", "destination_id", "trips"]
    missing = [col for col in required if col not in out.columns]
    if missing:
        raise KeyError(
            f"Missing OD columns: {missing}. "
            f"Available columns: {out.columns.tolist()}"
        )

    out["origin_id"] = standardize_zone_id(out["origin_id"])
    out["destination_id"] = standardize_zone_id(out["destination_id"])

    trips = pd.to_numeric(out["trips"], errors="coerce")
    if trips.isna().any():
        raise ValueError(
            f"Trip column contains {trips.isna().sum():,} missing or invalid values."
        )
    if trips.lt(0).any():
        raise ValueError("Trip column contains negative values.")
    out["trips"] = trips.astype(float)

    out["OD_pair"] = (
        out["origin_id"].astype(str)
        + "_"
        + out["destination_id"].astype(str)
    )

    return out


def compute_manuscript_srmse(
    hts_share: np.ndarray,
    lcs_share: np.ndarray,
) -> float:
    """Compute SRMSE as RMSE divided by the mean LCS share."""
    hts_share = np.asarray(hts_share, dtype=float)
    lcs_share = np.asarray(lcs_share, dtype=float)

    valid = np.isfinite(hts_share) & np.isfinite(lcs_share)
    if not valid.any():
        return np.nan

    y_pred = hts_share[valid]
    y_true = lcs_share[valid]

    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))

    # Retained for numerical consistency with the reported analysis.
    return float(rmse / (np.mean(y_true) + 1e-10))


def build_overall_spatial_srmse(
    hts_df: pd.DataFrame,
    lcs_df: pd.DataFrame,
    axis: str,
) -> pd.DataFrame:
    """Compute overall Origin-SRMSE or Destination-SRMSE."""
    hts_df = standardize_od_table(hts_df)
    lcs_df = standardize_od_table(lcs_df)

    key_cols = ["OD_pair", "origin_id", "destination_id"]

    hts = (
        hts_df[key_cols + ["trips"]]
        .rename(columns={"trips": "trips_hts"})
        .copy()
    )
    lcs = (
        lcs_df[key_cols + ["trips"]]
        .rename(columns={"trips": "trips_lcs"})
        .copy()
    )

    merged = hts.merge(
        lcs,
        on=key_cols,
        how="outer",
    )

    merged["trips_hts"] = merged["trips_hts"].fillna(0.0)
    merged["trips_lcs"] = merged["trips_lcs"].fillna(0.0)

    if axis == "origin":
        spatial_col = "origin_id"
        counterpart_col = "destination_id"
        metric_col = "Origin_SRMSE"
    elif axis == "destination":
        spatial_col = "destination_id"
        counterpart_col = "origin_id"
        metric_col = "Destination_SRMSE"
    else:
        raise ValueError("axis must be either 'origin' or 'destination'.")

    results = []

    for spatial_id, group in merged.groupby(
        spatial_col,
        sort=True,
        dropna=False,
    ):
        total_hts = group["trips_hts"].sum()
        total_lcs = group["trips_lcs"].sum()

        share_hts = np.where(
            total_hts > 0,
            group["trips_hts"] / total_hts,
            0.0,
        )
        share_lcs = np.where(
            total_lcs > 0,
            group["trips_lcs"] / total_lcs,
            0.0,
        )

        results.append({
            spatial_col: spatial_id,
            metric_col: compute_manuscript_srmse(
                share_hts,
                share_lcs,
            ),
            "HTS_total_trips_in_unit": float(total_hts),
            "LCS_total_trips_in_unit": float(total_lcs),
            "n_od_cells": int(len(group)),
            "n_positive_lcs_cells": int((share_lcs > 0).sum()),
            "n_unique_counterpart_zones": int(
                group[counterpart_col].nunique(dropna=True)
            ),
        })

    result = (
        pd.DataFrame(results)
        .sort_values(spatial_col)
        .reset_index(drop=True)
    )

    if len(result) != N_SEOUL_ZONES:
        raise ValueError(
            f"Expected {N_SEOUL_ZONES} spatial units, "
            f"but found {len(result)}."
        )

    if not result["n_od_cells"].eq(N_SEOUL_ZONES).all():
        raise ValueError(
            "Each spatial unit must contain exactly "
            f"{N_SEOUL_ZONES} counterpart OD cells."
        )

    return result


## 2. Construct Origin-SRMSE and Destination-SRMSE

- **Origin-SRMSE** compares the destination-trip distributions of HTS and LCS within each origin.
- **Destination-SRMSE** compares the origin-trip distributions of HTS and LCS within each destination.


In [ ]:
OD_HTS = read_csv_flexible(OD_HTS_FILE)
OD_LCS = read_csv_flexible(OD_LCS_FILE)

ORIGIN_SRMSE = build_overall_spatial_srmse(
    hts_df=OD_HTS,
    lcs_df=OD_LCS,
    axis="origin",
)

DESTINATION_SRMSE = build_overall_spatial_srmse(
    hts_df=OD_HTS,
    lcs_df=OD_LCS,
    axis="destination",
)

ORIGIN_SRMSE.to_csv(
    ORIGIN_SRMSE_FILE,
    index=False,
    encoding="utf-8-sig",
)
DESTINATION_SRMSE.to_csv(
    DESTINATION_SRMSE_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("Save complete")
# print(f"Saved: {ORIGIN_SRMSE_FILE}")
# print(f"Saved: {DESTINATION_SRMSE_FILE}")

display(ORIGIN_SRMSE.head())
display(DESTINATION_SRMSE.head())


## 3. Construct regression-ready origin and destination tables

In [ ]:
def clean_income_quintile(series: pd.Series) -> pd.Categorical:
    """Convert labels such as '8분위' to ordered income-class labels."""
    income_num = pd.to_numeric(
        series.astype(str).str.extract(r"(\d+)", expand=False),
        errors="coerce",
    )

    labels = pd.Series(pd.NA, index=series.index, dtype="string")
    valid = income_num.notna()
    labels.loc[valid] = (
        income_num.loc[valid].astype(int).astype(str) + "_class"
    )

    categorical = pd.Categorical(
        labels,
        categories=INCOME_LEVELS,
        ordered=True,
    )

    if pd.isna(categorical).any():
        raise ValueError(
            "Income quintile contains missing or unexpected categories."
        )

    return categorical


def load_regression_attributes(
    file_path: Path,
    selected_cols: list[str],
    rename_map: dict[str, str],
    id_col: str,
) -> pd.DataFrame:
    """Load and standardize the explanatory-variable table."""
    source = read_csv_flexible(file_path)

    missing = [col for col in selected_cols if col not in source.columns]
    if missing:
        raise KeyError(
            f"Missing columns in {file_path.name}: {missing}"
        )

    df = source[selected_cols].rename(columns=rename_map).copy()
    df[id_col] = standardize_zone_id(df[id_col])
    df["income_quintile"] = clean_income_quintile(
        df["income_quintile"]
    )

    non_numeric = {id_col, "income_quintile"}
    numeric_cols = [
        col for col in df.columns
        if col not in non_numeric
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if df[col].isna().any():
            raise ValueError(
                f"{file_path.name}: {col} contains missing "
                "or non-numeric values."
            )

    if df[id_col].duplicated().any():
        raise ValueError(
            f"{file_path.name} contains duplicated {id_col} values."
        )

    return df.sort_values(id_col).reset_index(drop=True)


def load_srmse_metric(
    file_path: Path,
    id_col: str,
    metric_col: str,
) -> pd.DataFrame:
    """Load the zone ID and SRMSE column needed for regression."""
    df = read_csv_flexible(file_path)

    required = [id_col, metric_col]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(
            f"Missing columns in {file_path.name}: {missing}"
        )

    out = df[required].copy()
    out[id_col] = standardize_zone_id(out[id_col])
    out[metric_col] = pd.to_numeric(
        out[metric_col],
        errors="coerce",
    )

    if out[metric_col].isna().any():
        raise ValueError(
            f"{file_path.name} contains missing {metric_col} values."
        )

    if out[id_col].duplicated().any():
        raise ValueError(
            f"{file_path.name} contains duplicated {id_col} values."
        )

    return out.sort_values(id_col).reset_index(drop=True)


In [ ]:
# Origin-level explanatory variables
origin_selected_cols = [
    "424_행정구역코드",
    "평균공시지가(원/㎡)",
    "가구수",
    "소득분위",
    "단위면적당버스정류장개수",
    "토지이용비율_주거",
    "토지이용비율_상업",
    "토지이용비율_공업",
    "토지이용비율_녹지",
    "1차산업_사업체규모(명/개)",
    "2차산업_사업체규모(명/개)",
    "3차산업_사업체규모(명/개)",
    "분산",
    "총 인구수",
    "참여율",
]

origin_rename_map = {
    "424_행정구역코드": "origin_id",
    "평균공시지가(원/㎡)": "average_land_price",
    "가구수": "number_of_households",
    "소득분위": "income_quintile",
    "단위면적당버스정류장개수":
        "number_of_bus_stops_per_unit_area",
    "토지이용비율_주거": "residential_land_use",
    "토지이용비율_상업": "commercial_land_use",
    "토지이용비율_공업": "industrial_land_use",
    "토지이용비율_녹지": "green_space_land_use",
    "1차산업_사업체규모(명/개)":
        "business_size_primary_industry",
    "2차산업_사업체규모(명/개)":
        "business_size_secondary_industry",
    "3차산업_사업체규모(명/개)":
        "business_size_tertiary_industry",
    # Precomputed variance-based measure of destination-trip distribution
    "분산": "variation_in_od_destinations",
    "총 인구수": "total_population",
    "참여율": "survey_participation_rate",
}

# Destination-level explanatory variables
destination_selected_cols = [
    "424_행정구역코드",
    "평균공시지가(원/㎡)",
    "가구수",
    "소득분위",
    "단위면적당버스정류장개수",
    "토지이용비율_주거",
    "토지이용비율_상업",
    "토지이용비율_공업",
    "토지이용비율_녹지",
    "1차산업_사업체규모(명/개)",
    "2차산업_사업체규모(명/개)",
    "3차산업_사업체규모(명/개)",
    "분산",
    "참여율",
]

destination_rename_map = {
    "424_행정구역코드": "destination_id",
    "평균공시지가(원/㎡)": "average_land_price",
    "가구수": "number_of_households",
    "소득분위": "income_quintile",
    "단위면적당버스정류장개수":
        "number_of_bus_stops_per_unit_area",
    "토지이용비율_주거": "residential_land_use",
    "토지이용비율_상업": "commercial_land_use",
    "토지이용비율_공업": "industrial_land_use",
    "토지이용비율_녹지": "green_space_land_use",
    "1차산업_사업체규모(명/개)":
        "business_size_primary_industry",
    "2차산업_사업체규모(명/개)":
        "business_size_secondary_industry",
    "3차산업_사업체규모(명/개)":
        "business_size_tertiary_industry",
    # Precomputed variance-based measure of origin-trip distribution
    "분산": "variation_in_od_origins",
    "참여율": "survey_participation_rate",
}

x_origin = load_regression_attributes(
    file_path=ORIGIN_ATTR_FILE,
    selected_cols=origin_selected_cols,
    rename_map=origin_rename_map,
    id_col="origin_id",
)

x_destination = load_regression_attributes(
    file_path=DESTINATION_ATTR_FILE,
    selected_cols=destination_selected_cols,
    rename_map=destination_rename_map,
    id_col="destination_id",
)


In [ ]:
origin_srmse = load_srmse_metric(
    file_path=ORIGIN_SRMSE_FILE,
    id_col="origin_id",
    metric_col="Origin_SRMSE",
)

destination_srmse = load_srmse_metric(
    file_path=DESTINATION_SRMSE_FILE,
    id_col="destination_id",
    metric_col="Destination_SRMSE",
)

df_origin = x_origin.merge(
    origin_srmse,
    on="origin_id",
    how="left",
    validate="1:1",
)

df_destination = x_destination.merge(
    destination_srmse,
    on="destination_id",
    how="left",
    validate="1:1",
)

if len(df_origin) != N_SEOUL_ZONES:
    raise ValueError(
        f"Origin regression table must contain {N_SEOUL_ZONES} zones."
    )

if len(df_destination) != N_SEOUL_ZONES:
    raise ValueError(
        f"Destination regression table must contain {N_SEOUL_ZONES} zones."
    )

if df_origin["Origin_SRMSE"].isna().any():
    raise ValueError("Missing Origin_SRMSE values after the merge.")

if df_destination["Destination_SRMSE"].isna().any():
    raise ValueError("Missing Destination_SRMSE values after the merge.")

df_origin.to_csv(
    ORIGIN_REGRESSION_FILE,
    index=False,
    encoding="utf-8-sig",
)
df_destination.to_csv(
    DESTINATION_REGRESSION_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("Save complete")
# print(f"Saved: {ORIGIN_REGRESSION_FILE}")
# print(f"Saved: {DESTINATION_REGRESSION_FILE}")

display(df_origin.head())
display(df_destination.head())


## 4. PCA and OLS regression

The regression procedure follows the former Appendix C(5):

- the four land-use proportions are standardized before PCA;
- one land-use principal component is retained;
- continuous main-effect variables are standardized;
- interaction terms are formed **after** standardization;
- income classes 5–9 are included as dummy variables;
- `10_class` is explicitly fixed as the reference category;
- ordinary least squares is estimated with conventional standard errors.

The origin interaction model corresponds to **Table 5**.  
The destination interaction model corresponds to **Appendix Table B.2**.


In [ ]:
LAND_USE_VARS = [
    "residential_land_use",
    "commercial_land_use",
    "industrial_land_use",
    "green_space_land_use",
]

COMMON_NUMERIC_VARS = [
    "average_land_price",
    "number_of_households",
    "number_of_bus_stops_per_unit_area",
    "business_size_primary_industry",
    "business_size_secondary_industry",
    "business_size_tertiary_industry",
    "survey_participation_rate",
]


def prepare_regression_designs(
    df: pd.DataFrame,
    dependent_col: str,
    variation_col: str,
) -> dict:
    """Build baseline and interaction design matrices."""
    required = (
        LAND_USE_VARS
        + COMMON_NUMERIC_VARS
        + [variation_col, dependent_col, "income_quintile"]
    )

    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"Missing regression columns: {missing}")

    model_df = df.copy()

    numeric_cols = (
        LAND_USE_VARS
        + COMMON_NUMERIC_VARS
        + [variation_col, dependent_col]
    )

    for col in numeric_cols:
        model_df[col] = pd.to_numeric(
            model_df[col],
            errors="coerce",
        )

    if model_df[numeric_cols].isna().any().any():
        missing_counts = (
            model_df[numeric_cols]
            .isna()
            .sum()
            .loc[lambda x: x.gt(0)]
            .to_dict()
        )
        raise ValueError(
            f"Regression variables contain missing values: {missing_counts}"
        )

    # PCA on standardized land-use variables
    pca_scaler = StandardScaler()
    land_use_scaled = pca_scaler.fit_transform(
        model_df[LAND_USE_VARS]
    )

    pca = PCA(n_components=1)
    model_df["PCA_landuse"] = pca.fit_transform(
        land_use_scaled
    )[:, 0]

    pca_info = pd.DataFrame({
        "PCA_Component": ["PCA_landuse_1"],
        "Explained_Variance": pca.explained_variance_,
        "Explained_Variance_Ratio":
            pca.explained_variance_ratio_,
        "Cumulative_Explained_Variance":
            pca.explained_variance_ratio_.cumsum(),
    })

    pca_loadings = pd.DataFrame({
        "Variable": LAND_USE_VARS,
        "PCA_landuse_1": pca.components_[0],
    })

    # Income dummies with the manuscript reference category fixed explicitly
    income = pd.Categorical(
        model_df["income_quintile"].astype("string"),
        categories=INCOME_LEVELS,
        ordered=True,
    )

    if pd.isna(income).any():
        raise ValueError(
            "Income quintile contains values outside the expected "
            f"categories: {INCOME_LEVELS}"
        )

    income_dummies = pd.get_dummies(
        income,
        dtype=float,
    )

    if INCOME_REFERENCE not in income_dummies.columns:
        raise ValueError(
            f"Reference category {INCOME_REFERENCE} is missing."
        )

    income_dummies = income_dummies.drop(
        columns=[INCOME_REFERENCE]
    )
    income_dummies.index = model_df.index

    numeric_main_effects = [
        "average_land_price",
        "number_of_households",
        "number_of_bus_stops_per_unit_area",
        "business_size_primary_industry",
        "business_size_secondary_industry",
        "business_size_tertiary_industry",
        variation_col,
        "survey_participation_rate",
        "PCA_landuse",
    ]

    main_scaler = StandardScaler()
    scaled_main = pd.DataFrame(
        main_scaler.fit_transform(
            model_df[numeric_main_effects]
        ),
        columns=numeric_main_effects,
        index=model_df.index,
    )

    interaction_main = scaled_main.copy()
    interaction_main["households_x_participation"] = (
        interaction_main["number_of_households"]
        * interaction_main["survey_participation_rate"]
    )
    interaction_main["tertiary_business_x_variation_od"] = (
        interaction_main["business_size_tertiary_industry"]
        * interaction_main[variation_col]
    )

    X_baseline = pd.concat(
        [scaled_main, income_dummies],
        axis=1,
    ).astype(float)

    X_interaction = pd.concat(
        [interaction_main, income_dummies],
        axis=1,
    ).astype(float)

    y = model_df[dependent_col].astype(float)

    return {
        "X_baseline": X_baseline,
        "X_interaction": X_interaction,
        "y": y,
        "pca_info": pca_info,
        "pca_loadings": pca_loadings,
    }


def fit_ols(X: pd.DataFrame, y: pd.Series):
    """Fit OLS with an intercept and conventional standard errors."""
    X_with_const = sm.add_constant(
        X,
        has_constant="add",
    )
    return sm.OLS(y, X_with_const).fit()


def model_coefficient_table(model) -> pd.DataFrame:
    """Convert a statsmodels result to a reusable coefficient table."""
    conf_int = model.conf_int()

    return pd.DataFrame({
        "variable": model.params.index,
        "coefficient": model.params.values,
        "std_error": model.bse.values,
        "t_value": model.tvalues.values,
        "p_value": model.pvalues.values,
        "ci_lower_95": conf_int.iloc[:, 0].values,
        "ci_upper_95": conf_int.iloc[:, 1].values,
    })


### 4.1 Origin-level models

In [ ]:
origin_design = prepare_regression_designs(
    df=df_origin,
    dependent_col="Origin_SRMSE",
    variation_col="variation_in_od_destinations",
)

origin_baseline_model = fit_ols(
    origin_design["X_baseline"],
    origin_design["y"],
)

origin_interaction_model = fit_ols(
    origin_design["X_interaction"],
    origin_design["y"],
)

print("PCA component summary")
display(origin_design["pca_info"].round(4))

print("PCA loadings")
display(origin_design["pca_loadings"].round(4))

print(f"Income reference category: {INCOME_REFERENCE}")

print("\n[Origin baseline model]")
print(origin_baseline_model.summary())

print("\n[Origin interaction model: Table 5]")
print(origin_interaction_model.summary())


### 4.2 Destination-level models

In [ ]:
destination_design = prepare_regression_designs(
    df=df_destination,
    dependent_col="Destination_SRMSE",
    variation_col="variation_in_od_origins",
)

destination_baseline_model = fit_ols(
    destination_design["X_baseline"],
    destination_design["y"],
)

destination_interaction_model = fit_ols(
    destination_design["X_interaction"],
    destination_design["y"],
)

print("PCA component summary")
display(destination_design["pca_info"].round(4))

print("PCA loadings")
display(destination_design["pca_loadings"].round(4))

print(f"Income reference category: {INCOME_REFERENCE}")

print("\n[Destination baseline model]")
print(destination_baseline_model.summary())

print("\n[Destination interaction model: Appendix Table B.2]")
print(destination_interaction_model.summary())


## 5. Correlation and manuscript-value checks

In [ ]:
# Origin–destination SRMSE correlation
df_corr = (
    df_origin[["origin_id", "Origin_SRMSE"]]
    .rename(columns={"origin_id": "zone_id"})
    .merge(
        df_destination[
            ["destination_id", "Destination_SRMSE"]
        ].rename(columns={"destination_id": "zone_id"}),
        on="zone_id",
        how="inner",
        validate="1:1",
    )
)

corr_pearson = df_corr["Origin_SRMSE"].corr(
    df_corr["Destination_SRMSE"],
    method="pearson",
)

print(f"Number of matched zones: {len(df_corr)}")
print(f"Pearson correlation: {corr_pearson:.4f}")


def rounded_check(
    item: str,
    computed: float,
    expected: float,
    decimals: int,
) -> dict:
    """Compare a computed result with the manuscript at its display precision."""
    computed_rounded = round(float(computed), decimals)
    return {
        "item": item,
        "computed": computed_rounded,
        "manuscript": expected,
        "decimals": decimals,
        "status": (
            "OK"
            if computed_rounded == expected
            else "CHECK"
        ),
    }


checks = []

# PCA loadings reported in the manuscript
origin_loading_map = (
    origin_design["pca_loadings"]
    .set_index("Variable")["PCA_landuse_1"]
)

for variable, expected in {
    "residential_land_use": 0.7233,
    "commercial_land_use": -0.0869,
    "industrial_land_use": -0.3133,
    "green_space_land_use": -0.6092,
}.items():
    checks.append(
        rounded_check(
            f"PCA loading: {variable}",
            origin_loading_map.loc[variable],
            expected,
            4,
        )
    )

# Table 5: origin interaction model
for variable, expected in {
    "average_land_price": -0.322,
    "number_of_households": -0.421,
    "number_of_bus_stops_per_unit_area": 0.041,
    "business_size_primary_industry": 0.064,
    "business_size_secondary_industry": 0.040,
    "business_size_tertiary_industry": -0.386,
    "variation_in_od_destinations": 0.790,
    "survey_participation_rate": -0.154,
    "PCA_landuse": 0.100,
    "households_x_participation": -0.134,
    "tertiary_business_x_variation_od": -0.206,
}.items():
    checks.append(
        rounded_check(
            f"Table 5 coefficient: {variable}",
            origin_interaction_model.params[variable],
            expected,
            3,
        )
    )

checks.append(
    rounded_check(
        "Table 5 adjusted R-squared",
        origin_interaction_model.rsquared_adj,
        0.556,
        3,
    )
)

# Appendix Table B.2: destination interaction model
for variable, expected in {
    "average_land_price": -0.320,
    "number_of_households": -0.412,
    "number_of_bus_stops_per_unit_area": 0.042,
    "business_size_primary_industry": 0.065,
    "business_size_secondary_industry": 0.030,
    "business_size_tertiary_industry": -0.360,
    "variation_in_od_origins": 0.793,
    "survey_participation_rate": -0.172,
    "PCA_landuse": 0.092,
    "households_x_participation": -0.149,
    "tertiary_business_x_variation_od": -0.188,
}.items():
    checks.append(
        rounded_check(
            f"Table B.2 coefficient: {variable}",
            destination_interaction_model.params[variable],
            expected,
            3,
        )
    )

checks.append(
    rounded_check(
        "Table B.2 adjusted R-squared",
        destination_interaction_model.rsquared_adj,
        0.554,
        3,
    )
)

checks.append(
    rounded_check(
        "Origin–destination SRMSE correlation",
        corr_pearson,
        0.9785,
        4,
    )
)

MANUSCRIPT_CHECKS = pd.DataFrame(checks)
display(MANUSCRIPT_CHECKS)

if MANUSCRIPT_CHECKS["status"].ne("OK").any():
    print(
        "Warning: Some computed values do not match the manuscript "
        "at the reported precision."
    )
else:
    print(
        "All selected PCA, regression, and correlation values "
        "match the manuscript at the reported precision."
    )


## 6. Save regression outputs

In [ ]:
# PCA outputs
origin_design["pca_info"].to_excel(
    REGRESSION_RESULT_DIR / "Result_PCA_landuse_summary.xlsx",
    index=False,
)
origin_design["pca_loadings"].to_excel(
    REGRESSION_RESULT_DIR / "Result_PCA_landuse_loadings.xlsx",
    index=False,
)

destination_design["pca_info"].to_excel(
    REGRESSION_RESULT_DIR
    / "Result_PCA_landuse_summary_destination.xlsx",
    index=False,
)
destination_design["pca_loadings"].to_excel(
    REGRESSION_RESULT_DIR
    / "Result_PCA_landuse_loadings_destination.xlsx",
    index=False,
)

# Regression coefficient tables
model_coefficient_table(origin_baseline_model).to_csv(
    REGRESSION_RESULT_DIR
    / "Result_Regression_origin_baseline.csv",
    index=False,
    encoding="utf-8-sig",
)
model_coefficient_table(origin_interaction_model).to_csv(
    REGRESSION_RESULT_DIR
    / "Result_Regression_origin_interaction_Table5.csv",
    index=False,
    encoding="utf-8-sig",
)
model_coefficient_table(destination_baseline_model).to_csv(
    REGRESSION_RESULT_DIR
    / "Result_Regression_destination_baseline.csv",
    index=False,
    encoding="utf-8-sig",
)
model_coefficient_table(destination_interaction_model).to_csv(
    REGRESSION_RESULT_DIR
    / "Result_Regression_destination_interaction_TableB2.csv",
    index=False,
    encoding="utf-8-sig",
)

# Correlation and manuscript checks
df_corr.to_excel(
    REGRESSION_RESULT_DIR / "Result_Correlation.xlsx",
    index=False,
)
MANUSCRIPT_CHECKS.to_csv(
    REGRESSION_RESULT_DIR / "Result_Manuscript_Checks.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Save complete")
# print(f"All regression outputs saved under: {REGRESSION_RESULT_DIR}")


## Notes

- `Origin_SRMSE` is the dependent variable of the main regression reported in Table 5.
- `Destination_SRMSE` is used for the supplementary model reported in Appendix Table B.2.
- The source column `분산` is retained as a variance-based OD-distribution measure. Its upstream formula should be documented with the attribute-construction data or code.
- `total_population` is retained only as an audit field for the precomputed survey participation rate; it is not included in the reported regression model.
- Income class `10_class` is fixed explicitly as the reference category.
- The interaction terms are constructed after the continuous main effects are standardized, matching the former Appendix C(5) implementation.
